# GLM-4-Voice Audio Tokenizer Example

This notebook demonstrates the GLM-4-Voice audio tokenizer

## Setup and Imports


In [13]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


PyTorch version: 2.6.0a0+df5bbc09d1.nv24.11
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset


In [14]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [15]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize GLM-4-Voice Tokenizer


In [16]:
# Initialize the tokenizer
print("Loading GLM-4-Voice tokenizer...")
tokenizer = get_tokenizer('glm4voice', device=device, use_speaker_encoder=False)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: ~{tokenizer.sample_rate / tokenizer.downsample_rate:.1f} Hz (~12.5 tokens per second)")


Loading GLM-4-Voice tokenizer...


/users/lmantel/benchmark-audio-tokenizer/.venv-glm4voice/lib/python3.12/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/users/lmantel/benchmark-audio-tokenizer/examples/../src/repos/glm4voice/flow_inference.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/

Tokenizer loaded: GLM4VoiceTokenizer(checkpoint='None', device='cuda', sample_rate=16000)
  Input sample rate: 16000 Hz
  Output sample rate: 22050 Hz
  Codebook size: 4,096
  Downsample rate: 1280x
  Token rate: ~12.5 Hz (~12.5 tokens per second)


## Tokenize and Reconstruct Audio Samples


In [17]:
# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor
    audio_tensor = torch.from_numpy(audio_array).float()
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    if token_values.ndim > 1:
        # Handle multi-dimensional token arrays
        token_values = token_values.flatten()
    
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Try to decode back to audio
    try:
        print("\nDecoding tokens back to audio...")
        reconstructed, decode_info = tokenizer.decode(tokens)
        
        print(f"Reconstructed shape: {reconstructed.shape}")
        print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
        
        # Handle shape: (1, 1, T) or (1, T) or (T,)
        rec = reconstructed
        if rec.dim() == 3:
            rec = rec[0, 0] if rec.shape[1] == 1 else rec[0]
        elif rec.dim() == 2:
            rec = rec[0]
        rec_np = rec.detach().cpu().numpy() if torch.is_tensor(rec) else rec
        
        # Store results
        results.append({
            'original': audio_array,
            'original_sr': sr,
            'reconstructed': rec_np,
            'reconstructed_sr': decode_info['output_sample_rate'],
            'tokens': token_values,
            'transcript': transcript,
            'duration': duration
        })
    except Exception as e:
        print(f"⚠ Decoding not available: {e}")
        results.append({
            'original': audio_array,
            'original_sr': sr,
            'reconstructed': None,
            'reconstructed_sr': None,
            'tokens': token_values,
            'transcript': transcript,
            'duration': duration
        })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...


Token shape: torch.Size([1, 148])
Number of tokens: 148
Tokens per second: 12.6

First 20 discrete token IDs:
[10815 11402  8299  8392 10728 10099 15226  6709  8824  3847   949 14032
 13941  2052   359  7109  4840  6674 13934 14032]
Token ID range: [151, 16047]
Unique tokens used: 146

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 260864])
Output sample rate: 22050 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Token shape: torch.Size([1, 147])
Number of tokens: 147
Tokens per second: 12.6

First 20 discrete token IDs:
[10342  4819 13892  9540 15700  2079  1329  8417  1364 13229 12300 12300
 13104 13268 10592  5185  6752  9435   604  5655]
Token ID range: [130, 16340]
Unique tokens used: 140

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 259072])
Output sample rate: 2205

## Play Original vs Reconstructed Audio


In [18]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    if result['reconstructed'] is not None:
        print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
        display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    else:
        print("\n⚠ Reconstruction not available (decoder models not loaded)")
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 12-bit tokens (log2(4096) = 12)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 148 (12.6 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (22050 Hz):



Compression ratio: 1272.4x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 147 (12.6 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (22050 Hz):



Compression ratio: 1273.5x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 147 (12.6 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (22050 Hz):



Compression ratio: 1272.4x


## Summary Statistics


In [19]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")

# Calculate average compression ratio
avg_compression = np.mean([
    (len(r['original']) * 2) / (len(r['tokens']) * 2) 
    for r in results
])
print(f"Average compression ratio: {avg_compression:.1f}x")

print(f"\nGLM-4-Voice Properties:")
print(f"  Bitrate: {12.5 * 12 / 1000:.2f} kbps (approx)")
print(f"  Token rate: ~12.5 Hz")
print(f"  Bits per token: 12 (log2(4096))")
print(f"  Codebook size: 4,096")
print(f"  Input: 16 kHz mono")
print(f"  Output: 22.05 kHz mono")
print(f"  Architecture: Whisper-based encoder + Flow decoder")
print(f"  Voice cloning: Optional (via CAMPPlus speaker encoder)")


Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 442
Average tokens per second: 12.6
Average compression ratio: 1272.8x

GLM-4-Voice Properties:
  Bitrate: 0.15 kbps (approx)
  Token rate: ~12.5 Hz
  Bits per token: 12 (log2(4096))
  Codebook size: 4,096
  Input: 16 kHz mono
  Output: 22.05 kHz mono
  Architecture: Whisper-based encoder + Flow decoder
  Voice cloning: Optional (via CAMPPlus speaker encoder)
